# Notebook 01 -- word_list_scores

**Asks:** score every answer with the nine-facet word list.

**Reads:** `openends_long.csv` (notebook 00).
**Writes:** `facet_dictionary_scores.csv`.

The word list is the non-model scorer of the three. It reads no meaning: it counts
which of nine buckets of spelling patterns fire in each answer. That is its whole
value. Because it uses no language model, its agreement with the model ensemble is a
more independent check than the models' agreement with one another.

## Config

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "01"                     # this notebook's number
NB_NAME     = "word_list_scores"       # this notebook's job, in two words
NB_ASKS     = "Score every answer with the nine-facet word list."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 01 -- word_list_scores
  Score every answer with the nine-facet word list.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

In [4]:
# === LOAD: the long open-ends, one row per answer ===
# WHAT THIS CELL DOES: puts the long open-ends into `df`, the table every cell below
# scores, and prints the checks that prove the file is the one the pipeline expects.
# CONSUMES: openends_long.csv (notebook 00).
# PRODUCES: `df` -- one row per response_id x item_type, with `text` guaranteed to be
#           a string and `n_words` guaranteed to be a whole number. Writes nothing.

# re -> the nine facet buckets are regular expressions ("regex": a compact notation
#       for a family of spellings, so one line covers deserve / deserved / deserving).
# pandas -> everything here is table work.
import re
import pandas as pd

OPENENDS = need("openends_long.csv", produced_by="00")

# keep_default_na=False, matching the way notebook 00 wrote the file. A participant who
# literally answered "N/A" keeps that text instead of arriving as a missing value, which
# is what keeps `text` and `n_words` telling the same story about the same response and
# what lets the pattern scorer read what the person actually wrote.
df = pd.read_csv(OPENENDS, keep_default_na=False)

# ---- shape asserts: the contract this notebook depends on, checked before scoring ----
# issubset rather than equality, so a file carrying extra columns still passes.
expected_cols = {"response_id", "vignette", "item_type", "orientation", "text", "n_words"}
assert expected_cols.issubset(set(df.columns)), \
    f"Missing columns: {expected_cols - set(df.columns)}"
# Exact set equality here, deliberately: a missing item_type means a whole open-end
# went missing, and an extra one means something was concatenated or renamed. Either
# would silently change what E3 and E6 are computed on.
assert set(df.item_type.unique()) == {"ranking", "backward", "forward"}, \
    f"Unexpected item_type values: {sorted(df.item_type.unique())}"

# Belt and braces on top of keep_default_na=False: a blank open-end has to be an empty
# string rather than a missing value, because the scorer lowercases `text` directly and
# a missing value there would raise. astype(str) covers a column pandas typed as a
# number because every value was empty.
df["text"] = df["text"].fillna("").astype(str)
# n_words is written as a whole number by notebook 00. Coercing it again keeps a blank
# cell in a hand-edited file reading as 0 instead of turning the whole column into text,
# which every `n_words > 0` filter below depends on.
df["n_words"] = pd.to_numeric(df["n_words"], errors="coerce").fillna(0).astype(int)

if RUN_MODE == "smoke":
    # Plumbing check, never a result: 20 rows prove the chain runs end to end. The
    # tables printed below are meaningless at this size and the sparsity percentages
    # in particular must not be read.
    # Trim by PARTICIPANT, never by row. Every person contributes three rows, one
    # per open-ended question, and notebooks 00 and 03 both cut on participants.
    # A head(20) here would keep 20 ROWS, which is six and two thirds of a person,
    # and the three-rows-per-person check downstream would fail on the participant
    # left half in and half out.
    df = df[df["response_id"].isin(
        pd.unique(df["response_id"])[:20])].reset_index(drop=True)
    print("  RUN_MODE=smoke: scoring the first 20 rows only. Diagnostics are noise.")

# WHAT THESE COUNTS MEAN: all three items should be non-blank for very nearly every
# participant (the items use a soft prompt, prereg 5.15, so a few blanks are expected),
# and the length ordering should be ranking > backward > forward, median 32 / 19 / 13
# words. A collapsed count, or a median of 0 for one item, is the signature of a
# column-resolution failure upstream in notebook 00, not of a quiet sample.
print("\nNon-blank counts and median words per item_type:")
for it in ["ranking", "backward", "forward"]:
    sub = df[(df.item_type == it) & (df.n_words > 0)]
    med = int(sub.n_words.median()) if len(sub) else 0
    avg = sub.n_words.mean() if len(sub) else 0.0
    print(f"  {it:9s}: {len(sub):5d} non-blank, median {med} words (mean {avg:.1f})")

# Three rows per retained participant: 2,844 rows for the 948 responses that survive
# the preregistered exclusions (prereg 6.2).
print(f"\nLoaded {len(df)} rows to score.")

  [ok]   openends_long.csv  (written unknown time)

Non-blank counts and median words per item_type:
  ranking  :   945 non-blank, median 35 words (mean 44.3)
  backward :   947 non-blank, median 20 words (mean 25.8)
  forward  :   945 non-blank, median 14 words (mean 18.2)

Loaded 2844 rows to score.


## Do

### The refined nine-facet word list

Nine **disjoint** buckets of spelling patterns, emitting **raw counts, per-100-word
rates, and `n_words`**. Its role is the **non-model independent convergence check**
(prereg 6.7, safeguard 1) and *never a primary scorer*. Prereg 6.7 requires the
list's known false matches to be removed and no word to count toward two opposing
facets. Four rules carry that out:

- **The bare adverb `just` never counts toward desert.** Desert fires on
  `\b(un)?deserv\w*`, `\bjust deserts?`, `\bearn(ed|s)?\b`, `\bhad it coming`, and the
  like, never on `just` alone, which is an adverb ("he just got out") far more often
  than a desert claim.
- **Every pattern is word-bounded.** `\b` on both sides of `earn`, so `learn`, `yearn`
  and `relearn` cannot reach the desert bucket. "Learn his lesson" is rehabilitation
  language.
- **No token belongs to two opposing facets.** `\bremove\w* from society` sits only in
  `incapacitation`; the assertion below enforces disjointness for every
  principled/vengeful pair.
- **`pay for` is routed by its object.** `pay for (what|it|his|her|their)` goes to
  **DESERT** everywhere; `payback` and `make (him|her|them) pay` stay under **REVENGE**,
  as separate tokens.

In [5]:
# === THE WORD LIST, 1 of 7: what it is, and what it is not ===
# Each pattern is a regex matched on lowercased text with word boundaries.
#
# WHAT THIS CELL DOES: opens the nine-facet word list, which the six cells after it
# fill one bucket at a time. It is the non-model convergence check named in prereg
# 6.7 (safeguard 1).
# PRODUCES: FACETS, empty here and filled below. Writes nothing.
#
# ROLE, AND THE LIMIT ON IT: this list is a transparent, low-recall PRECISION check.
# It is never a primary scorer -- the LLM ensemble is (prereg 6.7). Its value is
# precisely that it uses no language model, so its agreement with the ensemble is a
# more independent check than the models' agreement with one another. Its cost is
# that it cannot read context, which is why the ambiguous phrasings are deliberately
# absent from it (see the routing notes below) and why stance and negation are
# delegated to the LLM rubric and the human coders (design 6).
#
# WHAT IS DELIBERATELY MISSING: the genuinely contestable phrasings. Bare "pay for"
# (as in "pay for the pain they caused") reads as revenge or suffering to many
# coders, so only the object-routed form is here. Hard-routing an ambiguous token
# identically across the dictionary, the LLM rubric and the human codebook would
# manufacture convergence: all three scorers would agree because all three were
# instructed the same way. Leaving it out keeps the word list an honest low-recall
# check and leaves the call to the two scorers that can read scope (design 2).
#
# WHY IT IS BUILT IN PIECES: one facet group per cell, in facet order, so a reader can
# find the desert patterns without scrolling past the revenge ones. The order the
# buckets are added in is the column order of everything downstream, so the buckets
# are added in F1-to-F9 order and never re-sorted.
FACETS = {}
print("Word list opened. The nine buckets are added one group at a time below.")

Word list opened. The nine buckets are added one group at a time below.


In [6]:
# === THE WORD LIST, 2 of 7: the precision rules ===
# THE PRECISION RULES prereg 6.7 requires, carried out here:
#   1. The bare adverb "just" counts toward nothing. It appears in exactly one
#      pattern, r"just deserts?", where it is bound to "deserts". Spoken English uses
#      "just" as an adverb ("he just got out", "just give him 20 years") far more
#      often than as a desert claim, so a bare token would fire on sentences with no
#      desert content at all and inflate the one facet the E4 test depends on.
#   2. No token counts toward two OPPOSING facets. r"\bremove\w* from society" sits
#      only in incapacitation; the assertion further below enforces this for every
#      principled x vengeful pair.
#   3. "pay for" is routed by its object: r"\bpay for (what|it|his|her|their)" -> DESERT
#      everywhere, while r"\bpayback", r"\bpay back" and r"\bmake (him|her|them) pay" stay
#      under REVENGE as separate tokens (design 2, fixed token routing).
#   4. Every pattern is WORD-BOUNDED on the left, and on the right wherever the suffix
#      is open, so a bucket fires on words rather than on the insides of words. \b on
#      both sides of "earn" is why "learn", "yearn" and "relearn" cannot reach the
#      desert bucket -- which matters more here than anywhere else, because "learn his
#      lesson" is REHABILITATION language and desert is one of the two facets E4 rests
#      on. The two prefixed forms the list does want are written out, \b(un)?deserv\w*
#      and \b(dis)?proportion\w*, so they are matched deliberately rather than by an
#      open left edge.
#   5. Within a bucket, no phrase fires two patterns. score_text() counts DISTINCT
#      PATTERNS, so one surface form has to have exactly one home: r"\bfits? the crime"
#      covers both "fit the crime" and "fits the crime" as one proportionality claim,
#      and the lookahead in the moral-balance debt pattern hands "pay his debt to
#      society" to r"\bdebt to society" alone.
print("Precision rules 1-5 are enforced by the patterns below and by the assertion")
print("cell that follows them. Nothing is computed here.")

Precision rules 1-5 are enforced by the patterns below and by the assertion
cell that follows them. Nothing is computed here.


In [7]:
# === THE WORD LIST, 3 of 7: F1 DESERT and F2 PROPORTIONALITY ===
FACETS.update({
    # F1 DESERT -- principled/bounded. The OFFENDER earned the punishment, as a
    # backward fact about what they did or are: agent-focused and
    # magnitude-unbounded. Survey validator: `deserved` (Retribution_1..4).
    # GATE-ELIGIBLE, and half of the E4 pair, so its precision matters most.
    #   \b(un)?deserv\w*  deserve / deserved / deserves / deserving / undeserved, the
    #                    "un" spelled out so the prefix is matched on purpose
    #   \bjust deserts?  the fixed phrase, the ONLY place the word "just" appears
    #   \bearn(ed|s)?\b  \b on both sides so "learn", "learns", "learned", "yearn" and
    #                    "relearn" cannot match, and neither can "earnest"/"earnings".
    #                    "learn his lesson" and "learn to control himself" are
    #                    rehabilitation language, the opposite facet, and desert is
    #                    where E3/E4 carry the most weight
    #   got what \w+ deserv   the \w+ absorbs the pronoun ("got what he deserved")
    #   \bpay for (...)  object-routed to desert; bare "pay for" is intentionally absent
    #   \baccountab\w*   "held accountable" is one of the commonest desert phrasings
    #                    in ordinary speech; design 2 registers it as a contestable
    #                    call, and a later notebook runs a sensitivity check that flips
    #                    it between desert and censure to show the gate result does not
    #                    hinge on it
    #   \breap(s|ed|ing)?\b   "reap what they sow", bounded so "reappear"/"reapply"
    #                    cannot match
    "desert": [
        r"\b(un)?deserv\w*", r"\bjust deserts?", r"\bearn(ed|s)?\b", r"\bhad it coming",
        r"\bgot what \w+ deserv", r"\bpay for (what|it|his|her|their)",
        r"\banswer for", r"\baccountab\w*", r"\btake responsib\w*", r"\breap(s|ed|ing)?\b",
    ],
    # F2 PROPORTIONALITY -- principled/bounded. Not whether punishment is owed but
    # HOW MUCH fits: a bounded calibration with an explicit ceiling or floor. Survey
    # validator: `proportionality_jd`. DESCRIPTIVE-ONLY (design 7) on a measured ~1.0%
    # corpus prevalence -- a facet present in one response in a hundred has near-zero
    # predictor variance and cannot yield a stable r >= .30. The preregistered gate
    # numbers are computed and reported anyway so the demotion is auditable rather
    # than assumed (registered deviation, design 0).
    #   \b(dis)?proportion\w*  proportional / proportionate / disproportionate, the
    #                    "dis" spelled out because overshoot is proportionality talk too
    #   \bfits? the crime   one pattern for both "fit the crime" and "fits the crime",
    #                    so the phrase is one proportionality claim and not two
    #   no more (than|and no less)  the Kantian ceiling stated positively
    #   too (harsh|leni\w+|much|little), excessive, overly harsh
    #                    the same ceiling stated as a complaint about overshoot,
    #                    which is how most participants actually phrase it
    "proportionality": [
        r"\b(dis)?proportion\w*", r"\bfits? the crime", r"\bfits? the offense",
        r"\bmatch(es|ed|ing)? the (crime|offense|severity)", r"\bcommensurate",
        r"\bappropriate (punishment|sentence|amount)", r"\bno more (than|and no less)",
        r"\btoo (harsh|leni\w+|much|little)", r"\bexcessive", r"\boverly harsh",
    ],
})
print(f"F1 desert and F2 proportionality added. Buckets so far: {list(FACETS)}")

F1 desert and F2 proportionality added. Buckets so far: ['desert', 'proportionality']


In [8]:
# === THE WORD LIST, 4 of 7: F3 CENSURE and F4 MORAL BALANCE ===
FACETS.update({
    # F3 CENSURE / DENUNCIATION -- principled/bounded. The social MESSAGE: punishment
    # as public condemnation, satisfiable symbolically. Survey validator: NONE
    # ADMISSIBLE. Prereg 5.15 names exactly four facet-to-scale mappings and censure
    # is not among them; the plausible proxies (society_reject, exclusion,
    # solidarity_durk, just_deserts) each correlate .48-.80 with `deserved` and would
    # launder the desert scale under a new name. So censure is reported as prevalence
    # plus agreement only and is DESCRIPTIVE-ONLY permanently (design 2).
    #   \bsend a message    bare, audience unspecified -- the expressive reading
    #   \bshow(s|ing)? that .{0,20}wrong   the {0,20} bound keeps
    #                       "shows that what he did was wrong" in while stopping the
    #                       pattern from spanning an entire sentence to reach a
    #                       distant "wrong"
    "censure": [
        r"\bsend a message", r"\bsend(s|ing)? a signal", r"\bdenounc\w*", r"\bcondemn\w*",
        r"\bsociety('s)? disapprov\w*", r"\bmorally wrong", r"\bwhat he did was wrong",
        r"\bunacceptable", r"\bconsequences for (his|her|their) actions",
        r"\bshow(s|ing)? that .{0,20}wrong",
    ],
    # F4 MORAL-BALANCE / DEBT -- principled/bounded. Restoring a LEDGER: righting the
    # wrong, paying a debt, closure. Kept apart from proportionality on purpose --
    # moral balance is about CLOSURE, proportionality about CALIBRATION, and the two
    # answer different questions about the same sentence. Survey validator: NONE
    # ADMISSIBLE (same reasoning as censure), so DESCRIPTIVE-ONLY permanently.
    #   \bpay (his|her|their|the|a|his/her)? ?debt(?! to society)
    #                    the optional possessive and the optional space let "pay debt",
    #                    "pay his debt" and "pay his/her debt" all match one pattern; the
    #                    lookahead hands "pay his debt to society" to r"\bdebt to society"
    #                    alone, so that phrase is one ledger claim rather than two, and
    #                    the leading \b keeps "repay their debt" with r"\brepay"
    "moral_balance": [
        r"\bdebt to society", r"\bpay (his|her|their|the|a|his/her)? ?debt(?! to society)",
        r"\brepay", r"\bbalance the scales?", r"\brestore (justice|balance|order|the balance)",
        r"\beven the score", r"\bset(ting)? things right", r"\brighting a wrong", r"\batone\w*",
    ],
})
print(f"F3 censure and F4 moral_balance added. Buckets so far: {list(FACETS)}")

F3 censure and F4 moral_balance added. Buckets so far: ['desert', 'proportionality', 'censure', 'moral_balance']


In [9]:
# === THE WORD LIST, 5 of 7: F5 REVENGE and F6 SUFFERING ===
FACETS.update({
    # F5 REVENGE / PAYBACK -- vengeful/unbounded. Retaliation, getting even, symmetric
    # hitting-back. Survey validator: `revenge` (revenge_1..4). GATE-ELIGIBLE, and the
    # other half of the E4 pair.
    #   payback, pay back, make (him|her|them) pay
    #                    the routing rule from the precision rules above: these are
    #                    REVENGE tokens and never desert ones, even though they share
    #                    the verb "pay" with the desert pattern. The object is what
    #                    separates them -- paying FOR WHAT YOU DID is a desert claim,
    #                    making someone PAY is a retaliation claim
    # BASE RATE: the dictionary fires revenge on ~0.7% of responses, well under the 5%
    # coverage floor, so the word list is DISQUALIFIED as an independent convergence
    # leg for revenge (design 6). That is why the preregistered E4 word-list
    # corroboration is reported as uninformative-by-base-rate, with its counts, rather
    # than as disconfirmation of E4 -- a statistic with no power cannot veto.
    "revenge": [
        r"\breveng\w*", r"\bvengean\w*", r"\bpayback", r"\bpay back", r"\bget back at",
        r"\bretaliat\w*", r"\ban eye for an eye", r"\bget even", r"\bsettle the score",
        r"\bmake (him|her|them) pay",
    ],
    # F6 SUFFERING-FOR-ITS-OWN-SAKE -- vengeful/unbounded. Wanting pain beyond any
    # instrumental aim. Survey validator: `suffering` (suffer_1, suffer_2), with
    # `degradation` and `harsh` as convergent secondaries. GATE-ELIGIBLE.
    # BASE RATE: ~2.5% -- borderline against the 5% coverage floor, so the word-list
    # leg for suffering is reported with its confidence interval rather than leaned on
    # (design 6).
    #   \bsuffer\w*      suffer / suffering / suffered / suffers
    #   \brot in         "rot in prison", the commonest unbounded-pain phrasing here
    #   \bagon\w*        agony / agonizing, the \b so "dragon", "wagon" and "paragon"
    #                    cannot match
    "suffering": [
        r"\bsuffer\w*", r"\bmake (him|her|them) (feel|hurt|miserable)", r"\brot in",
        r"\bdeserve to (suffer|hurt|feel pain)", r"\bfeel (pain|the pain)", r"\bagon\w*",
        r"\bharsh(est|ly)? (as possible|treatment)", r"\bmiserable",
    ],
})
print(f"F5 revenge and F6 suffering added. Buckets so far: {list(FACETS)}")

F5 revenge and F6 suffering added. Buckets so far: ['desert', 'proportionality', 'censure', 'moral_balance', 'revenge', 'suffering']


In [10]:
# === THE WORD LIST, 6 of 7: F7 DETERRENCE, F8 INCAPACITATION, F9 REHABILITATION ===
# The three consequentialist goals. They are scored and reported, and NEVER summed
# into any retributivism total (design 2). They exist so the retributive facets are
# measured against a full menu of punishment justifications rather than in a vacuum.
FACETS.update({
    # F7 DETERRENCE -- consequentialist. Preventing FUTURE crime by warning or
    # example.
    #   \bsend a message to others   the audience is what separates this from censure:
    #                    censure's message is "the act was wrong" (addressed to
    #                    society), deterrence's is "do not do this" (addressed to
    #                    would-be offenders). Multi-label scoring is the point --
    #                    a response that says both carries both facets.
    #   \bdeter(?!min)\w*   deter / deters / deterring / deterrent / deterrence /
    #                    determent, and the misspellings participants actually write
    #                    (deterant, deterent, deterrance). The lookahead keeps
    #                    "determine the sentence" out: settling on a number is
    #                    sentencing procedure, not a deterrence goal.
    #   \bso (others|people) (don'?t|won'?t)   the \b so "also people don't" cannot match
    "deterrence": [
        r"\bdeter(?!min)\w*", r"\bdiscourag\w*", r"\bsend a message to others",
        r"\bprevent(s|ing)? others", r"\bwarn(ing)? to others", r"\bthink twice",
        r"\bset an example", r"\bcrime doesn'?t pay", r"\bso (others|people) (don'?t|won'?t)",
    ],
    # F8 INCAPACITATION -- consequentialist. Protecting society by removing or
    # containing the offender.
    #   \bremove\w* from society   the token prereg 6.7 singles out. It lives HERE and
    #                    only here: removal is a forward-looking public-protection
    #                    claim, so filing it under a retributive facet would let a
    #                    purely consequentialist answer ("get him away from people")
    #                    read as retributive language and inflate exactly the facets
    #                    E3 and E4 are testing. The assertion cell below enforces the
    #                    exclusivity.
    #   \block(ed)? (up|away)    matches "lock up" and "locked away"; the \b so
    #                    "blocked up" and "clocked away" cannot match
    "incapacitation": [
        r"\boff the streets?", r"\bkeep\w* (him|her|them|society) safe", r"\bprotect society",
        r"\bprotect the public", r"\bcan'?t (hurt|harm) (anyone|others|again)",
        r"\bremove\w* from society", r"\bdanger to society",
        r"\bthreat to (society|others|the public)", r"\block(ed)? (up|away)",
    ],
    # F9 REHABILITATION -- consequentialist. Reform, treatment, a second chance.
    # Expected to peak on the forward-looking item, which is what makes the
    # backward-vs-forward register shift (E6) a validation of the scheme itself,
    # independent of the survey.
    #   \btreatment      the \b so "mistreatment" -- a complaint about how prisons treat
    #                    people -- stays out of the rehabilitation bucket
    "rehabilitation": [
        r"\brehabilitat\w*", r"\breform\w*", r"\bsecond chance",
        r"\bturn (his|her|their) life around", r"\breintegrat\w*", r"\bbecome a better",
        r"\bget help", r"\btreatment", r"\bcounsel\w*", r"\bproductive member",
        r"\blearn from",
    ],
})
print(f"F7-F9, the three consequentialist goals, added. All buckets: {list(FACETS)}")

F7-F9, the three consequentialist goals, added. All buckets: ['desert', 'proportionality', 'censure', 'moral_balance', 'revenge', 'suffering', 'deterrence', 'incapacitation', 'rehabilitation']


In [11]:
# === THE WORD LIST, 7 of 7: classes, column order, opposing pairs, compile ===
# The class groupings. PRINCIPLED and VENGEFUL are the two sides of E4's
# co-occurrence question; CONSEQUENTIALIST is a third axis, scored so the retributive
# facets are measured against a full menu of punishment justifications rather than in
# a vacuum. The consequentialist goals are NEVER summed into any retributivism total
# (design 2) -- these lists exist to define the opposing pairs and the any-flags, not
# to build a composite.
PRINCIPLED = ["desert", "proportionality", "censure", "moral_balance"]
VENGEFUL   = ["revenge", "suffering"]
CONSEQUENTIALIST = ["deterrence", "incapacitation", "rehabilitation"]
# Fixes the column order used everywhere below, so the scored CSV, the printed mean
# table and the agreement tables downstream all list the facets in the same order.
FACET_NAMES = list(FACETS.keys())

# Opposing-facet pairs whose lexicons must not share a token (double-count guard).
# The full 4 x 2 cross-product of PRINCIPLED x VENGEFUL, and nothing else: the
# consequentialist goals are a different axis, not an opposite pole, so a response can
# legitimately carry deterrence and desert at once and no disjointness is required
# there. Within these eight pairs a shared token would make a single phrase count
# toward both sides of the very correlation E4 estimates.
OPPOSING = [
    ("desert", "revenge"), ("desert", "suffering"),
    ("proportionality", "revenge"), ("proportionality", "suffering"),
    ("moral_balance", "revenge"), ("moral_balance", "suffering"),
    ("censure", "revenge"), ("censure", "suffering"),
]

# Compile once, reuse for every one of the ~2,800 rows. No re.IGNORECASE flag is set
# because score_text() lowercases the text before matching; the patterns are written
# lowercase throughout and carry their own boundaries where a boundary is needed.
COMPILED = {f: [re.compile(p) for p in pats] for f, pats in FACETS.items()}
assert len(FACETS) == 9, f"expected 9 facet buckets, found {len(FACETS)}"
print(f"{len(FACETS)} facet buckets compiled, in this column order:")
print(f"  {FACET_NAMES}")
print(f"  {sum(len(p) for p in FACETS.values())} patterns in total")

9 facet buckets compiled, in this column order:
  ['desert', 'proportionality', 'censure', 'moral_balance', 'revenge', 'suffering', 'deterrence', 'incapacitation', 'rehabilitation']
  86 patterns in total


In [12]:
# === DUPLICATE / OPPOSING-TOKEN ASSERTION ===
# Prereg 6.7: no word counts toward two opposing facets.
#
# WHAT THIS CELL DOES: enforces the disjointness rule programmatically and then says
# out loud what the rule does and does not establish. It writes nothing and changes
# nothing -- it either passes silently or stops the notebook.
#
# WHY IT IS AN ASSERT AND NOT A COMMENT: the lexicons are edited by hand, and a token
# pasted into a second bucket during an edit would be invisible in the output while
# quietly inflating both sides of the E4 correlation. The assertion turns that edit
# into a failure at the top of the notebook rather than a finding at the end of it.
for a, b in OPPOSING:
    # Set intersection over the PATTERN STRINGS of the two buckets.
    shared = set(FACETS[a]).intersection(FACETS[b])
    assert not shared, f"shared token between opposing facets {a}/{b}: {shared}"

# Also assert no token is literally duplicated across ANY two facets.
# Wider than the OPPOSING check: this one catches a pattern pasted into, say, both
# deterrence and incapacitation, which the pair list above does not cover because
# those two are not opposites. _seen maps each pattern to the first facet that
# claimed it; a second claim by a different facet raises.
_seen = {}
for f, pats in FACETS.items():
    for p in pats:
        if p in _seen and _seen[p] != f:
            raise AssertionError(f"token {p!r} appears in both {_seen[p]} and {f}")
        _seen[p] = f

# What the printed note MEANS, and the limit it states honestly:
# the comparison above is over regex SOURCE STRINGS, so it guarantees no single
# pattern is counted twice. It does not and cannot guarantee that two buckets never
# fire on the same text -- multi-label scoring makes co-firing legitimate and
# expected (prereg 5.15: more than one facet can apply to a response), and one
# bucket's pattern can be a substring of another's. Construct independence is
# therefore MEASURED, at the response level, in the facet correlation table computed
# later in the chain, and is never inferred from this assertion (anti-circularity
# rule 8).
print("Opposing-facet lexicons are disjoint; no duplicate tokens across facets.")
print()
print("NOTE: this is a DOUBLE-COUNT GUARD, not proof of construct independence.")
print("  It says nothing about whether two facets fire on the SAME texts. Real")
print("  independence is measured at the response level later in the chain (response-")
print("  level facet correlations). Contestable ambiguous tokens ('held accountable'")
print("  as bare desert, bare 'pay for the pain') are DELIBERATELY LEFT OUT of the")
print("  dictionary and resolved in context by the LLM + human coders, rather than")
print("  fiat-routed.")

Opposing-facet lexicons are disjoint; no duplicate tokens across facets.

NOTE: this is a DOUBLE-COUNT GUARD, not proof of construct independence.
  It says nothing about whether two facets fire on the SAME texts. Real
  independence is measured at the response level later in the chain (response-
  level facet correlations). Contestable ambiguous tokens ('held accountable'
  as bare desert, bare 'pay for the pain') are DELIBERATELY LEFT OUT of the
  dictionary and resolved in context by the LLM + human coders, rather than
  fiat-routed.


In [13]:
# === NEGATION, 1 of 2: the whitelist, and why it zeroes instead of flipping ===
# A raw proximity sign-flip corrupts counts: "deserves NO mercy" / "deserve no
# sympathy" are MAX-desert; "not only does he deserve it" does not negate desert.
# So we NEVER sign-flip. We only ZERO OUT exact whitelisted negation+cue spans that
# have been inspected case by case.
#
# Emit BOTH <facet> (raw) and <facet>_negzeroed columns. The neg-zeroed column is
# flagged "never feeds a gate" (anti-circularity rule 12, MEASUREMENT_DESIGN.md
# section 9).
#
# WHAT THIS CELL DOES: defines the whitelist. PRODUCES: NEG_ZERO (a dict). Writes
# nothing.
#
# THE FULL ARGUMENT, because this is where a well-meant edit does the most damage.
# The obvious implementation -- "if a negator appears within N words of a facet cue,
# flip the sign" -- gets its own target cases backwards:
#   "he deserves NO mercy"                 max-desert, would be flipped to negative
#   "no sympathy"                          max-desert, same
#   "NOT only does he deserve it"          intensifier, not a negation of desert
#   "I DON'T think a light sentence is deserved"   net pro-desert
# A rule that mishandles the constructions it exists for is worse than no rule, and a
# precision check has no business emitting negative counts at all. So:
#   - NEVER sign-flip. The counts stay non-negative.
#   - Zero out only EXACT spans that a human has read in context and whitelisted.
#   - Zero only the NAMED facet, never the whole row.
#   - Report raw and zeroed side by side, and keep the zeroed column out of every
#     gate computation (design 6; anti-circularity rule 12).
# Genuine stance and negation handling is delegated to the two scorers that can read
# scope: the LLM rubric (which is given a negation/stance anchor) and, where the
# optional human arm is run, the coders (whose instructions say so explicitly). The
# word list is disclaimed on stance rather than patched into competence it does not
# have.

# Each entry is an EXACT lowercased span that, where present, zeroes the hits for the
# named facet ONLY. Entries are added only after a human has read the response the
# span came from and confirmed the negation, one at a time, keyed to the single facet
# it negates. An empty whitelist is a correct state, not an unfinished one.
NEG_ZERO = {
    # "span (lowercased)": "facet_to_zero"
    # e.g. "don't want him to suffer": "suffering",
    #      "doesn't deserve mercy":     "desert",   # inspected: mercy-negation, not desert
    # NOTE: "deserve no mercy" is PRO-desert -> must NOT be listed here.
}
print(f"NEG_ZERO whitelist loaded ({len(NEG_ZERO)} spans).")
print("  Rule: neg-zeroed columns NEVER feed a gate (stance handling is delegated to")
print("  the LLM ensemble and, if it is run, the human arm).")

NEG_ZERO whitelist loaded (0 spans).
  Rule: neg-zeroed columns NEVER feed a gate (stance handling is delegated to
  the LLM ensemble and, if it is run, the human arm).


In [14]:
# === NEGATION, 2 of 2: the one function that applies the whitelist ===
# PRODUCES: negation_zeroed(). Writes nothing.
def negation_zeroed(text, raw_counts):
    """Return a copy of raw_counts with hits zeroed ONLY for facets whose exact
    whitelisted negation span appears in `text`. Never sign-flips; never touches
    a facet with no whitelisted span present."""
    # Pad with spaces so a whitelist entry written with leading/trailing spaces still
    # matches a span sitting at the very start or end of the response, and lowercase
    # to match the way the whitelist entries are written.
    t = " " + (text or "").lower() + " "
    # Copy, so the raw counts survive untouched and both columns can be written.
    out = dict(raw_counts)
    for span, facet in NEG_ZERO.items():
        # A plain substring test, not a regex. Every entry is an exact inspected
        # string, which is what makes the whitelist auditable line by line -- a reader
        # can check each span against the response it came from.
        if span in t and facet in out:
            out[facet] = 0
    return out

# With an empty whitelist every _negzeroed column equals its raw column, which is the
# correct behaviour and not a silent failure: the function is a no-op until a span is
# inspected and added above.
# Probe with NON-ZERO counts. An all-zero probe cannot fail: zeroing something
# already zero is a no-op, so a negation_zeroed() that wrongly zeroed EVERY facet
# on EVERY row would pass it. Starting from 1 is what makes this a real tripwire.
_demo = {f: 1 for f in FACET_NAMES}
assert negation_zeroed("he deserves no mercy", _demo) == _demo, \
    "negation_zeroed changed a row it had no whitelisted span for"
print("negation_zeroed() ready. With an empty whitelist it changes nothing, which is")
print("  the intended state until inspected spans are added.")

negation_zeroed() ready. With an empty whitelist it changes nothing, which is
  the intended state until inspected spans are added.


In [15]:
# === SCORE EVERY ROW, 1 of 2: the scorer ===
# WHAT THIS CELL DOES: defines the one function that turns a piece of text into nine
# counts. CONSUMES: COMPILED. PRODUCES: score_text(). Writes nothing.
def score_text(text):
    # The count is the number of DISTINCT PATTERNS in a bucket that fire at least
    # once, not the number of occurrences: rx.search() stops at the first match, and
    # the sum counts patterns. So a response saying "deserves" three times scores 1
    # on that pattern, while a response saying "deserves" and "had it coming" scores
    # 2. Padding and lowercasing here is what lets the patterns be written lowercase
    # and boundary-free.
    t = " " + (text or "").lower() + " "
    return {f: sum(1 for rx in pats if rx.search(t)) for f, pats in COMPILED.items()}

# A one-line self-check with a sentence whose answer is known by inspection: the
# desert bucket fires twice ("deserved" and "had it coming"), revenge not at all.
# Without it, a broken pattern edit would surface as a shifted mean forty lines later
# instead of as a failure here.
_probe = score_text("He deserved every year of it and he had it coming.")
assert _probe["desert"] == 2 and _probe["revenge"] == 0, f"scorer self-check failed: {_probe}"
print("score_text() ready; self-check on a known sentence passed.")

score_text() ready; self-check on a known sentence passed.


In [16]:
# === SCORE EVERY ROW, 2 of 2: the pass over the corpus ===
# CONSUMES: `df` (the long open-ends), COMPILED, NEG_ZERO.
# PRODUCES: `dict_df`, the word-list stream every later convergence check reads.
out_rows = []
# ~2,800 rows and 90-odd compiled patterns: iterrows is slow in principle and
# irrelevant in practice at this size, and it keeps the per-row record explicit.
for _, r in df.iterrows():
    raw = score_text(r["text"])
    # The whitelist copy. With an empty whitelist this equals `raw` for every row.
    nz  = negation_zeroed(r["text"], raw)
    # n_words arrives from notebook 00; the notna guard covers a hand-edited file.
    nw  = int(r["n_words"]) if pd.notna(r["n_words"]) else 0
    row = {
        # Carry every key the later analyses need: response_id x item_type is the
        # join key for the frozen split and for every other scoring stream,
        # orientation drives the E6 backward/forward contrast, vignette is needed
        # because facet scores are standardized within vignette x item_type wherever
        # vignettes are pooled, and n_words is the verbosity covariate.
        "response_id": r["response_id"], "item_type": r["item_type"],
        "orientation": r["orientation"], "vignette": r["vignette"], "n_words": nw,
    }
    for f in FACET_NAMES:
        # Three columns per facet, all three kept:
        #   <facet>            raw pattern-fire count
        #   <facet>_per100     the same count as a rate per 100 words. This is the
        #                      verbosity normalization: item medians run 32/19/13
        #                      words, so a raw count alone would make the ranking item
        #                      look the most retributive simply for being the longest.
        #                      Guarded to 0.0 on a blank response, where a rate is
        #                      undefined rather than zero.
        #   <facet>_negzeroed  the whitelist-zeroed count, reported beside the raw one
        #                      and barred from every gate computation (rule 12).
        row[f] = raw[f]
        row[f + "_per100"]    = round(100 * raw[f] / nw, 3) if nw else 0.0
        row[f + "_negzeroed"] = nz[f]          # NEVER feeds a gate
    # Any-hit indicators over the two retributive classes, from the RAW counts.
    # Descriptive convenience for quick prevalence reads and for the strata drawn
    # downstream; the E4 co-occurrence estimate uses the facet-level scores, never
    # these two flags, because collapsing four facets into one flag would throw away
    # which principled facet actually co-occurred with the vengeful one.
    row["principled_any"] = int(any(raw[f] > 0 for f in PRINCIPLED))
    row["vengeful_any"]   = int(any(raw[f] > 0 for f in VENGEFUL))
    out_rows.append(row)

dict_df = pd.DataFrame(out_rows)
print(f"Scored {len(dict_df)} rows on {len(FACET_NAMES)} facets "
      f"({dict_df.shape[1]} columns).")

Scored 2844 rows on 9 facets (34 columns).


In [17]:
# === DIAGNOSTIC 1: mean facet hits by item, raw and verbosity-normalized ===
# WHAT TO READ HERE: the register shift the scheme predicts (E6, design 3). The
# retributive facets should be highest on ranking and backward, the three
# consequentialist goals highest on forward. If that ordering is absent, the facet
# scheme is not picking up the backward/forward distinction it was built around --
# which would be a finding about the dictionary, not about the participants.
# Blank responses are excluded so the means are per response that said something.
print("Mean facet hits per response, by item_type:")
# Facet names truncated to 6 characters to keep the nine columns on one line.
hdr = "item      " + "".join(f"{f[:6]:>8}" for f in FACET_NAMES)
print(hdr)
for it in ["ranking", "backward", "forward"]:
    sub = dict_df[(dict_df.item_type == it) & (dict_df.n_words > 0)]
    if not len(sub):
        continue
    means = "".join(f"{sub[f].mean():8.2f}" for f in FACET_NAMES)
    print(f"{it:10s}{means}")

# ---- the same table, verbosity-normalized ----
# READ THIS ONE FOR E6, and the raw one above only for the counts behind it. The three
# items differ in length by more than a factor of two -- medians run about 34 words for
# ranking, 20 backward, 14 forward -- so a raw mean rewards the ranking item for being
# long before it says anything about register. The per-100-word columns are computed
# for exactly this reason, and the register shift the scheme predicts (retributive
# facets peaking on ranking/backward, the three consequentialist goals on forward) has
# to survive here, not only in the raw counts, to count as evidence.
print("\nMean facet hits per 100 words, by item_type (verbosity-normalized):")
print(hdr)
for it in ["ranking", "backward", "forward"]:
    sub = dict_df[(dict_df.item_type == it) & (dict_df.n_words > 0)]
    if not len(sub):
        continue
    rates = "".join(f"{sub[f + '_per100'].mean():8.2f}" for f in FACET_NAMES)
    print(f"{it:10s}{rates}")

Mean facet hits per response, by item_type:
item        desert  propor  censur  moral_  reveng  suffer  deterr  incapa  rehabi
ranking       0.34    0.01    0.01    0.00    0.01    0.00    0.40    0.23    0.55
backward      0.19    0.01    0.00    0.00    0.00    0.02    0.02    0.13    0.04
forward       0.10    0.02    0.02    0.01    0.01    0.03    0.07    0.06    0.11

Mean facet hits per 100 words, by item_type (verbosity-normalized):
item        desert  propor  censur  moral_  reveng  suffer  deterr  incapa  rehabi
ranking       0.83    0.01    0.01    0.01    0.03    0.01    0.93    0.61    1.59
backward      1.06    0.03    0.02    0.01    0.00    0.11    0.05    0.50    0.14
forward       0.77    0.08    0.14    0.04    0.04    0.23    0.46    0.46    1.44


In [18]:
# === DIAGNOSTIC 2: realized sparsity, the input to the coverage decision ===
# WHAT TO READ HERE: whether each bucket fired at all, and at roughly the rate the
# first tranche of 300 led us to expect. A bucket that has gone silent is a broken
# pattern, and this is where that shows up -- immediately, rather than as a missing
# convergence leg four notebooks later.
# Prevalence, not intensity: the share of non-blank responses with AT LEAST ONE hit.
# That is the quantity the coverage floor is stated in.
#
# THE DECISION ITSELF IS NOT MADE HERE. The >=5% minimum-coverage rule (design 6,
# registered as a deviation in design 0) says the word list counts as an INDEPENDENT
# CONVERGENCE LEG for a facet only where it fires on at least 5% of the relevant
# items. Below that, per-facet agreement rests on a handful of positive cases and is
# noise -- and worse, its silence reads as "no independent contradiction" when what
# it really means is "no independent measurement at all". That rule is applied per
# facet and per item, with counts and confidence intervals, in notebook 02, and a
# disqualified leg is reported there with its counts rather than dropped.
# HAZARD: edit FACETS above and you change these numbers, which changes which
# convergence legs notebook 02 admits. Re-read this table after any lexicon edit.
print("Share of non-blank responses with >=1 hit, by facet:")
print("  (first tranche: revenge ~0.7%, suffering ~2.5%, proportionality ~1.0%,")
print("   censure ~1.4%, moral_balance ~0.7% nonzero)")
nonblank = dict_df[dict_df.n_words > 0]
for f in FACET_NAMES:
    pct = 100 * (nonblank[f] > 0).mean() if len(nonblank) else 0.0
    print(f"  {f:16s} {pct:5.1f}% nonzero")
print()
print("The >=5% coverage floor is applied in NOTEBOOK 02, per facet and per item.")
if RUN_MODE == "smoke":
    print("\n  RUN_MODE=smoke: these percentages come from 20 rows and mean nothing.")
    print("  Nothing may be judged sparse on them.")

Share of non-blank responses with >=1 hit, by facet:
  (first tranche: revenge ~0.7%, suffering ~2.5%, proportionality ~1.0%,
   censure ~1.4%, moral_balance ~0.7% nonzero)
  desert            20.1% nonzero
  proportionality    1.2% nonzero
  censure            0.9% nonzero
  moral_balance      0.5% nonzero
  revenge            0.6% nonzero
  suffering          1.8% nonzero
  deterrence        15.3% nonzero
  incapacitation    13.9% nonzero
  rehabilitation    22.3% nonzero

The >=5% coverage floor is applied in NOTEBOOK 02, per facet and per item.


## Save and check

In [19]:
# === SAVE AND CHECK ===
# The one output of this notebook. Every later notebook that wants the word-list
# stream reads this file through need(), so it is written once, here, and recorded in
# the manifest.
DICT_OUT = "facet_dictionary_scores.csv"
# THE CONTRACT, ENFORCED ON THE FILE THIS NOTEBOOK WRITES. facet_dictionary_
# scores.csv is read by notebooks 02, 03, 04, 10 and 11, and notebooks 03 and
# 11 MERGE on (response_id, item_type). This is the exact file a checker
# duplicated to show the damage: three extra rows moved E4, E6 and E7 from 948
# participants to 951 with nothing raised and nothing printed. Notebook 02 also
# checks it on read, deliberately, because a file can be edited between the two
# notebooks. Both checks are cheap; the failure they catch is invisible.
assert not dict_df.duplicated(['response_id', 'item_type']).any(), (
    'facet_dictionary_scores.csv is NOT one row per participant per question. '
    'Duplicated keys: '
    + str(dict_df[dict_df.duplicated(['response_id', 'item_type'],
                                     keep=False)][['response_id',
                                                   'item_type']].head(6)))
dict_df.to_csv(DICT_OUT, index=False)
done(DICT_OUT, rows=len(dict_df),
     note="9 facet buckets; raw + per-100-word + negation-zeroed columns per facet")

print()
print("WHAT THIS FILE CONTAINS")
print("  one row per response_id x item_type, and for each of the nine facets:")
print("    <facet>            raw count of distinct patterns that fired")
print("    <facet>_per100     the same count per 100 words (verbosity-normalized)")
print("    <facet>_negzeroed  the whitelist-zeroed count -- NEVER feeds a gate")
print("  plus principled_any and vengeful_any, two descriptive class flags.")
print()
print("WHAT IT IS FOR, AND WHAT IT IS NOT FOR")
print("  FOR: the non-model leg of the convergence check (prereg 6.7, safeguard 1),")
print("       on the facets that clear the 5% coverage floor -- which facets those")
print("       are is decided in notebook 02, not here.")
print("  NOT FOR: any primary facet score. The LLM ensemble is the production")
print("       scorer. Where a facet falls below the floor, the word-list leg is")
print("       reported with its counts and marked disqualified, never left blank.")
print()
print("NEXT: notebook 02, which turns the prevalences printed above into the")
print("      per-facet coverage verdict the later convergence checks obey.")

  [wrote] facet_dictionary_scores.csv  (2844 rows, 333,872 bytes)

WHAT THIS FILE CONTAINS
  one row per response_id x item_type, and for each of the nine facets:
    <facet>            raw count of distinct patterns that fired
    <facet>_per100     the same count per 100 words (verbosity-normalized)
    <facet>_negzeroed  the whitelist-zeroed count -- NEVER feeds a gate
  plus principled_any and vengeful_any, two descriptive class flags.

WHAT IT IS FOR, AND WHAT IT IS NOT FOR
  FOR: the non-model leg of the convergence check (prereg 6.7, safeguard 1),
       on the facets that clear the 5% coverage floor -- which facets those
       are is decided in notebook 02, not here.
  NOT FOR: any primary facet score. The LLM ensemble is the production
       scorer. Where a facet falls below the floor, the word-list leg is
       reported with its counts and marked disqualified, never left blank.

NEXT: notebook 02, which turns the prevalences printed above into the
      per-facet coverage 